In [ ]:
# ================================
# Imports & paths (no installs, no unzip)
# ================================
!pip install ultralytics opencv-python tqdm

from ultralytics import YOLO
import os, shutil, cv2
import numpy as np

SRC_ROOT = "/content/extracted_organized_files/organized"      # root of your organized dataset
YOLO_ROOT = "/content/yolo_dataset"  # YOLO-formatted output
os.makedirs(YOLO_ROOT, exist_ok=True)

print("SRC_ROOT =", SRC_ROOT)
print("Images/train exists:", os.path.isdir(os.path.join(SRC_ROOT, "images", "train")))
print("Labels/train exists:", os.path.isdir(os.path.join(SRC_ROOT, "labels", "train")))

SRC_ROOT = /content/extracted_organized_files/organized
Images/train exists: True
Labels/train exists: True


In [ ]:
# ================================
# 1) Convert to YOLO format
# ================================

def read_label_lines(path):
    boxes = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 5:
                continue
            x1, y1, x2, y2 = map(float, parts[:4])
            cls_id = int(parts[4])
            boxes.append((x1, y1, x2, y2, cls_id))
    return boxes

def convert_box_to_yolo(x1, y1, x2, y2, w, h):
    cx = (x1 + x2) / 2.0 / w
    cy = (y1 + y2) / 2.0 / h
    bw = (x2 - x1) / w
    bh = (y2 - y1) / h
    return cx, cy, bw, bh

def process_split(split):
    src_img_dir = os.path.join(SRC_ROOT, "images", split)
    src_lbl_dir = os.path.join(SRC_ROOT, "labels", split)
    dst_img_dir = os.path.join(YOLO_ROOT, "images", split)
    dst_lbl_dir = os.path.join(YOLO_ROOT, "labels", split)

    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_lbl_dir, exist_ok=True)

    if not os.path.isdir(src_img_dir):
        print(f"[WARN] No images for split '{split}' in {src_img_dir}")
        return

    count_imgs = 0
    count_lbls = 0

    for name in os.listdir(src_img_dir):
        if not name.lower().endswith(".jpg"):
            continue

        img_src = os.path.join(src_img_dir, name)

        # 20085000_test.jpg -> 20085000.txt
        prefix = name.split("_")[0]
        prefix = os.path.splitext(prefix)[0]
        lbl_name = prefix + ".txt"
        lbl_src = os.path.join(src_lbl_dir, lbl_name)

        # copy image
        img_dst = os.path.join(dst_img_dir, name)
        if not os.path.exists(img_dst):
            shutil.copy2(img_src, img_dst)
        count_imgs += 1

        # negative sample: no label file
        if not os.path.exists(lbl_src):
            continue

        img = cv2.imread(img_src)
        if img is None:
            continue
        h, w = img.shape[:2]
        boxes = read_label_lines(lbl_src)
        if not boxes:
            continue

        yolo_lines = []
        for x1, y1, x2, y2, cls_id in boxes:
            cls_yolo = cls_id - 1  # 1..6 -> 0..5
            cx, cy, bw, bh = convert_box_to_yolo(x1, y1, x2, y2, w, h)
            cx = max(0.0, min(1.0, cx))
            cy = max(0.0, min(1.0, cy))
            bw = max(0.0, min(1.0, bw))
            bh = max(0.0, min(1.0, bh))
            yolo_lines.append(f"{cls_yolo} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        if not yolo_lines:
            continue

        lbl_dst = os.path.join(dst_lbl_dir, os.path.splitext(name)[0] + ".txt")
        with open(lbl_dst, "w") as f:
            f.write("\n".join(yolo_lines) + "\n")
        count_lbls += 1

    print(f"[{split}] images copied: {count_imgs}, labels written: {count_lbls}")

for split in ["train", "val", "test"]:
    process_split(split)

print("YOLO dataset root:", YOLO_ROOT)
!ls -R /content/yolo_dataset | head -n 40

[train] images copied: 900, labels written: 900
[val] images copied: 100, labels written: 100
[test] images copied: 500, labels written: 500
YOLO dataset root: /content/yolo_dataset
/content/yolo_dataset:
images
labels

/content/yolo_dataset/images:
test
train
val

/content/yolo_dataset/images/test:
00041200_test.jpg
00041201_test.jpg
00041202_test.jpg
00041203_test.jpg
00041204_test.jpg
00041205_test.jpg
00041206_test.jpg
00041207_test.jpg
00041208_test.jpg
00041209_test.jpg
00041210_test.jpg
00041211_test.jpg
00041212_test.jpg
00041213_test.jpg
00041214_test.jpg
00041215_test.jpg
00041216_test.jpg
00041217_test.jpg
00041218_test.jpg
00041219_test.jpg
00041220_test.jpg
12000001_test.jpg
12000017_test.jpg
12000038_test.jpg
12000052_test.jpg
12000068_test.jpg
12000085_test.jpg
12000102_test.jpg
12000119_test.jpg
12000136_test.jpg


In [ ]:
# ================================
# 2) Create YOLO data config
# ================================

yaml_text = """
path: /content/yolo_dataset
train: images/train
val: images/val
test: images/test

names:
  0: missing_hole
  1: mouse_bite
  2: open_circuit
  3: short_circuit
  4: spur
  5: spurious_copper
"""
with open("pcb_yolo.yaml", "w") as f:
    f.write(yaml_text)

print(open("pcb_yolo.yaml").read())


path: /content/yolo_dataset
train: images/train
val: images/val
test: images/test

names:
  0: missing_hole
  1: mouse_bite
  2: open_circuit
  3: short_circuit
  4: spur
  5: spurious_copper



In [ ]:
# ================================
# 3) Train YOLOv8-nano
# ================================

model = YOLO("yolov8n.pt")

model.train(
    data="pcb_yolo.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
)

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=pcb_yolo.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspective=0.0, plots=True, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d7ff23d0500>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [ ]:
# ================================
# 4) Inference + green / yellow / red PCB border
# ================================

CLASS_NON_FIXABLE = {0, 1}        # missing_hole, mouse_bite -> failed
CLASS_FIXABLE = {2, 3, 4, 5}      # open_circuit, short_circuit, spur, spurious_copper -> fixable
CONF_THRESH = 0.25

best_weights = "runs/detect/train/weights/best.pt"
model = YOLO(best_weights)

def classify_board(dets):
    has_non_fixable = any(int(cls) in CLASS_NON_FIXABLE for *_, cls in dets)
    has_fixable = any(int(cls) in CLASS_FIXABLE for *_, cls in dets)
    if has_non_fixable:
        return "failed", (0, 0, 255)      # red
    elif has_fixable:
        return "fixable", (0, 255, 255)   # yellow
    else:
        return "good", (0, 255, 0)        # green

def annotate_pcb(image_path, save_path=None, conf=CONF_THRESH):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(image_path)
    h, w = img.shape[:2]

    results = model.predict(source=img, conf=conf, verbose=False)[0]
    dets = results.boxes.data.cpu().numpy() if results.boxes is not None else np.zeros((0, 6))

    status, color = classify_board(dets)

    # PCB border
    cv2.rectangle(img, (5, 5), (w - 5, h - 5), color, 4)

    # Draw defect boxes
    for x1, y1, x2, y2, conf_score, cls_id in dets:
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = f"{int(cls_id)} {conf_score:.2f}"
        cv2.putText(img, label, (x1, max(10, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

    cv2.putText(img, status.upper(), (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2, cv2.LINE_AA)

    if save_path:
        cv2.imwrite(save_path, img)
    return img

# Example after training:
# test_img_path = "/content/organized/images/test/20085000_test.jpg"
# annotated = annotate_pcb(test_img_path, save_path="/content/annotated_test.jpg")
# from google.colab.patches import cv2_imshow
# cv2_imshow(annotated)

In [ ]:
# ================================
# 3) Train YOLOv8-nano
# ================================

model = YOLO("yolov8n.pt")

model.train(
    data="pcb_yolo.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
)

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=pcb_yolo.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspective=0.0, plots=True,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d7fdca71c70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [ ]:
# ================================
# 2) Create YOLO data config
# ================================

yaml_text = """
path: /content/yolo_dataset
train: images/train
val: images/val
test: images/test

names:
  0: missing_hole
  1: mouse_bite
  2: open_circuit
  3: short_circuit
  4: spur
  5: spurious_copper
"""
with open("pcb_yolo.yaml", "w") as f:
    f.write(yaml_text)

print(open("pcb_yolo.yaml").read())


path: /content/yolo_dataset
train: images/train
val: images/val
test: images/test

names:
  0: missing_hole
  1: mouse_bite
  2: open_circuit
  3: short_circuit
  4: spur
  5: spurious_copper



In [21]:
import shutil
import os

output_zip_path = '/content/yolo_dataset.zip'
shutil.make_archive('/content/yolo_dataset', 'zip', '/content/yolo_dataset')

print(f"Successfully created {output_zip_path}")
print("You can now download this file from the Colab file browser and upload its contents to your GitHub repository.")

Successfully created /content/yolo_dataset.zip
You can now download this file from the Colab file browser and upload its contents to your GitHub repository.


In [19]:
print('Contents of the model weights directory:')
!ls -lh /content/runs/detect/train2/weights/

Contents of the model weights directory:
total 12M
-rw-r--r-- 1 root root 6.0M Feb 11 16:14 best.pt
-rw-r--r-- 1 root root 6.0M Feb 11 16:14 last.pt


In [ ]:
%%writefile .gitignore
# Python bytecode
__pycache__/
*.pyc

# Jupyter Notebook checkpoints
.ipynb_checkpoints/

# Ultralytics runs directory (excluding specific weights if desired)
# The 'runs' directory can be quite large, so we usually only keep the essential model weights.
# You can manually select 'best.pt' and 'last.pt' from your last training run (e.g., /content/runs/detect/train/weights/) to upload.
runs/

# YOLO dataset cache files
*.cache

# Local environment variables
.env

# Any other temporary or generated files
*.zip
*.log
*.txt
*.yaml


Writing .gitignore


The `.gitignore` file will prevent unnecessary files from being committed to your GitHub repository. Now, you can download the required files from your Colab environment to your local machine, and then upload them to your GitHub repository. Make sure to download:

*   The entire `/content/yolo_dataset` directory.
*   The `pcb_yolo.yaml` file.
*   The trained model weights: `/content/runs/detect/train2/weights/best.pt` and `/content/runs/detect/train2/weights/last.pt`.

In [ ]:
# Load the best model weights
best_weights = "/content/runs/detect/train2/weights/best.pt"
model = YOLO(best_weights)

# Run validation on the test set to get metrics
metrics = model.val(data="pcb_yolo.yaml", split="test")

# Print overall accuracy metrics
print("\nOverall Accuracy Metrics:")
print(f"Precision (all classes): {metrics.results_dict['metrics/precision(B)']:.3f}")
print(f"Recall (all classes): {metrics.results_dict['metrics/recall(B)']:.3f}")
print(f"mAP50 (all classes): {metrics.results_dict['metrics/mAP50(B)']:.3f}")
print(f"mAP50-95 (all classes): {metrics.results_dict['metrics/mAP50-95(B)']:.3f}")

# Plot and save confusion matrix
print("\nConfusion Matrix (saved to runs/detect/val/confusion_matrix.png):")
metrics.confusion_matrix.plot(save_dir="/content/runs/detect/val")

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1202.9±513.0 MB/s, size: 40.8 KB)
val: Scanning /content/yolo_dataset/labels/test.cache... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 209.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 5.2it/s 6.2s
                   all        500       3140      0.953       0.94      0.973      0.736
          missing_hole        482        659      0.966      0.957      0.981      0.667
            mouse_bite        368        478      0.936      0.839      0.916      0.624
          open_circuit        413        586      0.986      0.954      0.987      0.716
         short_circuit        373        483      0.976      0.932      0.977      0.724
                  spur        403        464 

After executing the code above, the accuracy metrics for the model will be displayed, and a confusion matrix will be generated and saved. The confusion matrix provides a detailed breakdown of correct and incorrect classifications for each defect class.

In [ ]:
import zipfile
import os

zip_file_path = '/content/organized (2).zip'
extract_dir = '/content/extracted_organized_files'

# Create the directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"Successfully extracted '{zip_file_path}' to '{extract_dir}'")
    print("Contents of extracted directory:")
    for root, dirs, files in os.walk(extract_dir):
        for name in files:
            print(os.path.join(root, name))
        for name in dirs:
            print(os.path.join(root, name))
except zipfile.BadZipFile:
    print(f"Error: '{zip_file_path}' is not a valid zip file or is corrupted.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Successfully extracted '/content/organized (2).zip' to '/content/extracted_organized_files'
Contents of extracted directory:
/content/extracted_organized_files/organized
/content/extracted_organized_files/organized/by_defect
/content/extracted_organized_files/organized/images
/content/extracted_organized_files/organized/labels
/content/extracted_organized_files/organized/by_defect/defect_classes_summary.txt
/content/extracted_organized_files/organized/by_defect/class_6
/content/extracted_organized_files/organized/by_defect/class_5
/content/extracted_organized_files/organized/by_defect/class_1
/content/extracted_organized_files/organized/by_defect/class_3
/content/extracted_organized_files/organized/by_defect/class_4
/content/extracted_organized_files/organized/by_defect/class_2
/content/extracted_organized_files/organized/by_defect/class_6/images
/content/extracted_organized_files/organized/by_defect/class_6/labels
/content/extracted_organized_files/organized/by_defect/class_6/images/v